In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the raw CRDC Chronic Absenteeism data
input_path = '../data/raw/ID 814 SCH - Chronic Absenteeism.csv'
df = pd.read_csv(input_path)
df.head()

,LEA_STATE,LEA_STATE_NAME,NCESLEAID,LEA_NAME,SCHID,SCHOOL_NAME,NCESSCH,JJ,AM_M_7,AS_M_7,...,TOTAL_STUDENTS_REPORTED_M,TOTAL_STUDENTS_REPORTED_F,WDIS_M,WDIS_F,DISAB504_M,DISAB504_F,LEP_M,LEP_F,H_M,H_F
0,AL,ALABAMA,100005,Albertville City,870,Albertville Middle School,10000500870,No,-8,-8,...,61,36,8,2,-8,-8,3,4,4,3
1,AL,ALABAMA,100005,Albertville City,871,Albertville High School,10000500871,No,-8,-8,...,110,126,15,2,-8,-8,15,17,1,2
2,AL,ALABAMA,100005,Albertville City,879,Evans Elementary School,10000500879,No,-8,-8,...,44,44,4,4,-8,-8,-8,4,-8,-8
3,AL,ALABAMA,100005,Albertville City,889,Albertville Elementary School,10000500889,No,-8,-8,...,54,57,10,6,-8,-8,6,3,4,-8
4,AL,ALABAMA,100005,Albertville City,1616,Big Spring Lake Kindergarten School,10000501616,No,-8,2,...,43,36,15,3,-8,-8,6,3,1,1


In [3]:
# Load the raw CRDC Enrollment data
input_path2 = '../data/raw/Enrollment.csv'
df2 = pd.read_csv(input_path2)
df2.head()

/var/folders/3s/m47q3wjj27l75f4v048s8kl40000gn/T/ipykernel_50789/2444897938.py:3: DtypeWarning: Columns (2,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv(input_path2)


,LEA_STATE,LEA_STATE_NAME,LEAID,LEA_NAME,SCHID,SCH_NAME,COMBOKEY,JJ,SCH_PSENR_NONIDEA_A3,SCH_PSENR_NONIDEA_A4,...,SCH_504ENR_BL_M,SCH_504ENR_BL_F,SCH_504ENR_WH_M,SCH_504ENR_WH_F,SCH_504ENR_TR_M,SCH_504ENR_TR_F,TOT_504ENR_M,TOT_504ENR_F,SCH_504ENR_LEP_M,SCH_504ENR_LEP_F
0,AL,ALABAMA,100002,Alabama Youth Services,1705,Wallace Sch - Mt Meigs Campus,10000201705,Yes,-9,-9,...,0,0,0,0,0,0,0,0,0,0
1,AL,ALABAMA,100002,Alabama Youth Services,1706,McNeel Sch - Vacca Campus,10000201706,Yes,-9,-9,...,0,0,0,0,0,0,0,0,0,0
2,AL,ALABAMA,100002,Alabama Youth Services,1876,Alabama Youth Services,10000201876,No,-9,-9,...,0,0,0,0,0,0,0,0,0,0
3,AL,ALABAMA,100002,Alabama Youth Services,99995,AUTAUGA CAMPUS,10000299995,Yes,-9,-9,...,0,0,0,0,0,0,0,0,0,0
4,AL,ALABAMA,100005,Albertville City,870,Albertville Middle School,10000500870,No,-9,-9,...,0,0,0,4,0,0,0,4,0,0


### Data Sources & Scope

This notebook joins two 2017-18 CRDC/EDFacts files: `ID 814` (chronic absenteeism) and `Enrollment.csv`. Initially, I assumed `TOTAL_STUDENTS_REPORTED_M`/`_F` represented total number of enrolled students, but investigation revealed total number of chronically absent students. I confirmed this by summing the totals and comparing against ~8.12M, a known national figure of ~8M.

For enrollment, I used `TOT_ENR_M`/`_F` rather than `TOT_PSENR_M`/`_F` because I did not want to include preschool enrollment.

**Scope note:** this analysis includes K-12, because I wanted to investigate whether district's overall climate of chronic absenteeism at all grade levels predicts graduation outcomes. A more targeted high-school-only version would require 9-12, which I deferred because of time constraints and undefined grade-span rule.

In [4]:
# Check shape of each dataframe
print('Raw Shape of Chronic Absenteeism:', df.shape)
print('Raw Shape of Enrollment:', df2.shape)

Raw Shape of Chronic Absenteeism: (92437, 32)
Raw Shape of Enrollment: (97632, 123)


In [5]:
# Rename IDs for consistency
df = df.rename(columns={
    'NCESLEAID': 'LEAID',
})

In [6]:
# Checking dtypes for identified columns
print(df[["TOTAL_STUDENTS_REPORTED_M", "TOTAL_STUDENTS_REPORTED_F"]].dtypes)
print(df2['COMBOKEY'].dtypes)

TOTAL_STUDENTS_REPORTED_M    int64
TOTAL_STUDENTS_REPORTED_F    int64
dtype: object
object


In [7]:
# Convert to numeric; drop negative codes used in documentation
for x in ["TOTAL_STUDENTS_REPORTED_M", "TOTAL_STUDENTS_REPORTED_F"]:
    df[x] = pd.to_numeric(df[x], errors='coerce')
    df.loc[df[x] < 0, x] = np.nan

# Convert to numeric; California independently-chartered LEAs (510 schools) use alphanumeric key
for c in ['COMBOKEY']:
    df2[c] = pd.to_numeric(df2[c], errors='coerce')

In [8]:
# Merge Absenteeism and Enrollment dataframes
merged_df = pd.merge(
    df,
    df2,
    left_on='NCESSCH',
    right_on='COMBOKEY',
    how='inner'
)

print(merged_df.head())
print(merged_df.shape)

  LEA_STATE_x LEA_STATE_NAME_x  LEAID_x        LEA_NAME_x  SCHID_x  \
0          AL          ALABAMA   100005  Albertville City      870   
1          AL          ALABAMA   100005  Albertville City      871   
2          AL          ALABAMA   100005  Albertville City      879   
3          AL          ALABAMA   100005  Albertville City      889   
4          AL          ALABAMA   100005  Albertville City     1616   

                           SCHOOL_NAME      NCESSCH JJ_x  AM_M_7  AS_M_7  ...  \
0            Albertville Middle School  10000500870   No      -8      -8  ...   
1              Albertville High School  10000500871   No      -8      -8  ...   
2              Evans Elementary School  10000500879   No      -8      -8  ...   
3        Albertville Elementary School  10000500889   No      -8      -8  ...   
4  Big Spring Lake Kindergarten School  10000501616   No      -8       2  ...   

   SCH_504ENR_BL_M  SCH_504ENR_BL_F  SCH_504ENR_WH_M  SCH_504ENR_WH_F  \
0                0 

### Join Key Resolution

Both files are at the school grain. Neither `LEAID` alone nor `SCHID` alone uniquely identifies a school: `LEAID` only identifies school districts (not a specific school within it), and `SCHID` only identifies individual schools (not which district it's in — the same SCHID number can appear in different LEAs).

Instead, I joined on `NCESSCH`/`COMBOKEY` — a single column on each side that already encodes school and school district combined into one value. I used an inner join, because I wanted to keep schools that were present in both files.

**Dtype mismatch:** `NCESSCH` was `int64` dtype, `COMBOKEY` was `object` dtype. This was caused by 510 rows in `Enrollment.csv` where `COMBOKEY` contained alphanumeric codes instead of a plain number — these represented California independently-chartered LEAs (a real-world category of school/district). I resolved this by coercing `COMBOKEY` to numeric, which meant losing 510 rows from `Enrollment.csv` before the join. I chose this over converting those values to string (the alternative: converting `NCESSCH` to match instead) because I wanted to avoid converting the data to string and then encoding.

**Duplicate columns:** after the merge, `LEAID`, `SCHID`, `LEA_STATE`, `LEA_STATE_NAME`, `LEA_NAME`, and `JJ` all existed in both files and came through suffixed `_x` (from df) and `_y` (from df2). I kept the `_x` versions because they contained int64.

Separately, later in the pipeline, I found and dropped 10 rows where `chron_absent` was known but `enrolled` was missing — a different, smaller issue from the `COMBOKEY` dtype problem above.

In [9]:
# Rename and drop duplicate columns
suffix_cols = [c for c in merged_df.columns if c.endswith('_x') or c.endswith('_y')]
print(suffix_cols)

merged_df = merged_df.rename(columns={
    'LEA_STATE_x': 'LEA_STATE',
    'LEA_STATE_NAME_x': 'LEA_STATE_NAME',
    'LEAID_x': 'LEAID',
    'LEA_NAME_x': 'LEA_NAME',
    'SCHID_x': 'SCHID',
    'JJ_x': 'JJ'
})

merged_df = merged_df.drop(columns=[
    'LEA_STATE_y', 'LEA_STATE_NAME_y', 'LEAID_y', 'LEA_NAME_y', 'SCHID_y', 'JJ_y' 
])

['LEA_STATE_x', 'LEA_STATE_NAME_x', 'LEAID_x', 'LEA_NAME_x', 'SCHID_x', 'JJ_x', 'LEA_STATE_y', 'LEA_STATE_NAME_y', 'LEAID_y', 'LEA_NAME_y', 'SCHID_y', 'JJ_y']


In [10]:
# Keep necessary columns
keep_cols = [
    'LEAID', 'SCHID', 'SCH_NAME', 'TOTAL_STUDENTS_REPORTED_M', 'TOTAL_STUDENTS_REPORTED_F', 'TOT_ENR_M', 'TOT_ENR_F']

merged_df = merged_df[keep_cols].copy()

In [11]:
# Convert to numeric; drop negative codes used in documentation
for y in ['TOT_ENR_M', 'TOT_ENR_F']:
    merged_df[y] = pd.to_numeric(merged_df[y], errors='coerce')
    merged_df.loc[merged_df[y] < 0, y] = np.nan

chron_absent_cols = ['TOTAL_STUDENTS_REPORTED_M', 'TOTAL_STUDENTS_REPORTED_F']
enroll_cols = ['TOT_ENR_M', 'TOT_ENR_F']

merged_df['chron_absent'] = merged_df[chron_absent_cols].sum(axis=1, min_count=1)
merged_df['enrolled'] = merged_df[enroll_cols].sum(axis=1, min_count=1)

In [12]:
# Drop rows where 'chron_absent' is known but 'enrolled' is missing
mismatched = merged_df[merged_df['chron_absent'].notna() & merged_df['enrolled'].isna()]
print(f'Dropping {len(mismatched)} rows: known absent count but missing enrollment')
merged_df = merged_df.dropna(subset=['enrolled'])

Dropping 10 rows: known absent count but missing enrollment


### Numerator/Denominator Integrity Check

After computing `chron_absent` and `enrolled` per school, I checked whether either could be known while the other was missing. I found 10 rows (all belonging to a single Massachusetts district) where chronic absenteeism was known but enrollment was missing. I dropped these rows rather than include them, because keeping a known school with absenteeism without its matching total student population would skew the dataset.

In [13]:
# Groupby to LEA level with min_count
lea_df = merged_df.groupby(
    'LEAID', as_index=False).agg(
        chron_absent=('chron_absent', lambda x: x.sum(min_count=1)),
        enrolled=('enrolled', lambda x: x.sum(min_count=1))
    )

In [14]:
# Calculate chronic absenteeism rates at the district level
lea_df['absent_rate'] = (lea_df['chron_absent'] / lea_df['enrolled']) * 100
lea_df['absent_rate'].describe()

count    15998.000000
mean        17.975106
std         41.898199
min          0.000000
25%          8.180942
50%         12.969248
75%         20.089325
max       4466.666667
Name: absent_rate, dtype: float64

In [15]:
# Confirming 'absent_rate' data above 100% threshold
print('Sum of absent_rate < 0:', (lea_df['absent_rate'] < 0).sum())
print('Sum of absent_rate > 100:', (lea_df['absent_rate'] > 100).sum())
print('Sum of absent_rate inf:', np.isinf(lea_df['absent_rate']).sum())
print('Sum of absent_rate is NaN:', lea_df['absent_rate'].isna().sum())

Sum of absent_rate < 0: 0
Sum of absent_rate > 100: 188
Sum of absent_rate inf: 0
Sum of absent_rate is NaN: 0


In [16]:
# Exclude high-mobility/alternative schools — not confirmed for all cases
lea_df = lea_df[lea_df['absent_rate'] <= 100]
print(lea_df['absent_rate'].describe())
print('Sum of absent_rate > 100:', (lea_df['absent_rate'] > 100).sum())

count    15810.000000
mean        16.053023
std         12.720343
min          0.000000
25%          8.130694
50%         12.858727
75%         19.733993
max        100.000000
Name: absent_rate, dtype: float64
Sum of absent_rate > 100: 0


### Absent Rate > 100% Investigation

188 districts had `absent_rate` over 100% — impossible, since that means more chronically absent students than enrolled. Ruled out a merge bug: `NCESSCH` had 0 duplicates before and after the join.

`LEAID 4280070`, the worst case, is a single school: "Reading Muhlenberg CTC" (Career and Technical Center) — a vocational/technical school that many students attend part-time while still being "enrolled" at their home high school.

`LEAID 4800207` (33 schools) has names like "PREMIER H S SAN ANTONIO EAST" and "...OF BROWNSVILLE" — a charter network, likely operating dozens of small campuses across Texas as an alternative-education network, which could signal high student turnover.

I excluded these 188 districts (vs. capping at 100) because including these districts would introduce an inaccurate number of districts that have 100% chronic absenteeism, which would be misleading. This is 1.18% of districts; the national weighted rate barely moved (15.87% -> 15.75%), suggesting the overall dataset was not meaningfully impacted.

In [17]:
# Save processed .csv file
output_path = '../data/processed/crdc_lea_absenteeism_2017_18_v2.csv'
lea_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')

Saved to ../data/processed/crdc_lea_absenteeism_2017_18_v2.csv
